# Assignment 3 - IMDB Sentiment Analysis (Complete Workflow)

In [ ]:

import pandas as pd
df = pd.read_csv('IMDB Dataset.csv')
print(df.shape)
df.head()


In [ ]:

TEXT_COL = 'review'
TARGET_COL = 'sentiment'

print(df[TARGET_COL].value_counts())


In [ ]:

import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>',' ',text)
    text = re.sub(r'[^a-zA-Z\s]',' ',text)
    text = re.sub(r'\s+',' ',text).strip()
    return text


In [ ]:

df['clean_review'] = df[TEXT_COL].apply(clean_text)
df[['clean_review']].head()


In [ ]:

from sklearn.model_selection import train_test_split

X = df['clean_review']
y = df[TARGET_COL]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)


In [ ]:

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow = CountVectorizer(max_features=5000)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [ ]:

from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
    'MultinomialNB': MultinomialNB(),
    'BernoulliNB': BernoulliNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'LinearSVC': LinearSVC(),
    'RandomForest': RandomForestClassifier(n_estimators=100),
    'KNN': KNeighborsClassifier()
}

results = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)

    results.append([
        name,
        accuracy_score(y_test,pred),
        f1_score(y_test,pred,pos_label='positive')
    ])

pd.DataFrame(results, columns=['Model','Accuracy','F1'])


In [ ]:

# Hyperparameter Tuning Example

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {'C':[0.1,1,10]},
    cv=5,
    scoring='accuracy'
)

# grid.fit(X_train_tfidf, y_train)


In [ ]:

# Visualization 1 - Class Distribution

sns.countplot(x=y)
plt.show()


In [ ]:

# Visualization 2 - Review Length Distribution

lengths = X.str.len()
plt.hist(lengths, bins=30)
plt.show()


In [ ]:

# Visualization 3 - Model Comparison

# results_df = pd.DataFrame(results, columns=['Model','Accuracy','F1'])
# results_df.plot(x='Model', y='Accuracy', kind='bar')
# plt.show()



## Remaining Assignment Tasks
- N-gram experiments (1,2,3 grams)
- Character n-grams
- ROC Curves
- Precision Recall Curves
- Confusion Matrices
- Feature Engineering
- Chi-square Feature Selection
- Mutual Information
- Error Analysis
- Ensemble Model (Bonus)
- Save Models (.pkl)
